In [5]:
import numpy as np
from molsim import MonteCarlo, blockAverage
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import sys

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Exercise 2: Widom insertion

The **chemical potential** $\mu$ is a key thermodynamic quantity that characterizes how the free energy of a system changes when particles are added or removed. In an $NVT$ ensemble (fixed number of particles $N$, volume $V$, and temperature $T$), measuring $\mu$ can be challenging if one tries to explicitly remove or insert particles as part of the simulation. Instead, a clever “alchemical” approach—**Widom insertion**—provides a route to measure $\mu$ without permanently altering the number of particles in the system.

### Core Idea of Widom Insertion

1. **Test insertion**: During the simulation, you occasionally and virtually (“hypothetically”) insert an additional particle into the system at random positions (and orientations if the particle is non-spherical). You do *not* keep this test particle in the system; instead, you merely compute the energy change that *would* have been incurred if you were to insert it.

2. **Statistical averaging**: By repeating many such test insertions, you build up a statistical average of the Boltzmann factor $\exp(-\beta \Delta U)$, where $\Delta U$ is the energy change associated with inserting the test particle and $\beta = 1 / (k_B T)$. This average relates directly to the *excess* chemical potential $\mu^\mathrm{ex}$.

3. **Connection to total chemical potential**: The chemical potential $\mu$ in the $NVT$ ensemble comprises two parts:
   $
   \mu = \mu^\mathrm{id} + \mu^\mathrm{ex},
   $
   where $\mu^\mathrm{id}$ is the *ideal-gas* contribution, and $\mu^\mathrm{ex}$ is the *excess* part arising from interactions between the particles. Widom’s insertion method primarily determines $\mu^\mathrm{ex}$.

---

## Mathematical Formulation

### Excess Chemical Potential via Widom Insertion

For a system of $N$ particles in a volume $V$ at temperature $T$, Widom insertion tells us that the excess chemical potential $\mu^\mathrm{ex}$ is given by:

$
\beta \mu^\mathrm{ex} \;=\; -\ln \left\langle \exp(-\beta \Delta U) \right\rangle_{N,V,T},
$

where:

- $\beta = \frac{1}{k_B T}$,
- $\Delta U$ is the (hypothetical) energy change upon inserting an extra test particle into the system,
- $\langle \cdots \rangle_{N,V,T}$ denotes an ensemble average in the $NVT$ ensemble.

In practice, you perform a large number of test insertions during the simulation. For each test insertion $i$, you randomly choose a position (and orientation, if applicable) in the simulation box, compute the energy change $\Delta U_i$, and then accumulate the quantity $\exp(-\beta \Delta U_i)$. After $M$ such test insertions, you estimate

$
\left\langle \exp(-\beta \Delta U) \right\rangle \;\approx\; \frac{1}{M} \sum_{i=1}^M \exp(-\beta \Delta U_i).
$

Hence,

$
\beta \mu^\mathrm{ex} \;\approx\; - \ln \left( \frac{1}{M} \sum_{i=1}^M \exp(-\beta \Delta U_i) \right).
$

And the *excess* chemical potential is

$
\mu^\mathrm{ex} \;=\; - k_B T \,\ln \left( \frac{1}{M} \sum_{i=1}^M \exp(-\beta \Delta U_i) \right).
$


<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

## Question 1
Now we will probe the chemical potential of a simple Lennard-Jones simulation. Compute $\mu_0$ and make a plot of the excess chemical potential and pressure as a function of the density for $T= 0.8$. Use the densities given in the cell below. 

What are the thermodynamic conditions required vapour-liquid coexistence densities?

In [ ]:
numberOfParticles = 100
numberOfSteps = int(5e4)
densities = np.array(
    [
        0.001,
        0.003,
        0.006,
        0.009,
        0.01,
        0.02,
        0.03,
        0.04,
        0.05,
        0.06,
        0.07,
        0.08,
        0.09,
        0.1,
        0.2,
        0.3,
        0.4,
        0.5,
        0.6,
        0.7,
        0.8,
        0.9,
        1.0,
    ]
)

chemicalPotentials = np.zeros_like(densities)
chemicalPotentialConfidence = np.zeros_like(densities)
pressures = np.zeros_like(densities)

for i, density in enumerate(tqdm(densities)):
    sys.stdout.flush()
    mc = MonteCarlo(
        numberOfParticles=numberOfParticles,
        temperature=0.8,
        boxSize=np.cbrt(numberOfParticles / density),
        maxDisplacement=0.5,
        numberOfInitCycles=numberOfSteps,
        numberOfProdCycles=numberOfSteps,
        sampleFrequency=100,
        seed=120,
        logLevel=1,
    )

    # Run the Monte Carlo simulation
    mc.run()

    chemicalPotentials[i] = np.mean(mc.totalChemicalPotentials)
    pressures[i] = np.mean(mc.pressures)

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

Make the plots below. How can you locate the vapour-liquid coexistence densities?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].errorbar(densities, chemicalPotentials, c="red", marker="o", capsize=2)
ax[0].set_xlabel(r"Density, $\rho$ / $\sigma^{-3}$")
ax[0].set_ylabel(r"Chemical Potential / $\varepsilon$")

ax[1].plot(densities, pressures, c="red", marker="o")
ax[1].set_xlabel(r"Density, $\rho$ / $\sigma^{-3}$")
ax[1].set_ylabel(r"Pressure, p / $\varepsilon \sigma^{-3}$")

# start refactor
ax[2].plot([], [], c='red', marker='o')
# end refactor
ax[2].set_xlim(-5.6, -2.5)
ax[2].set_ylim(-0.75, 0.6)

fig.tight_layout()

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

## Question 2
 Have a look at the code in `molsim/monteCarlo/mc.cpp` in the function `MonteCarlo::computeChemicalPotential`, where the chemical potential is measured. Something interesting is happening in the function `MonteCarlo::logThermodynamicalAverages`, namely there is a block averaging happening of the Widom weights calculated. 
 
 Why are block averages useful?

 Implement the block averaging method to calculate the error of the chemical potential.

 Perform a simulation at high density and analyze the difference between the error at high and low number of trial positions.


In [ ]:
numberOfParticles = 100
numberOfSteps = int(5e4)
density = 0.1

chemicalPotentials = np.zeros_like(densities)
chemicalPotentialConfidence = np.zeros_like(densities)
pressures = np.zeros_like(densities)


mc = MonteCarlo(
    numberOfParticles=numberOfParticles,
    temperature=0.8,
    boxSize=np.cbrt(numberOfParticles / density),
    maxDisplacement=0.5,
    numberOfInitCycles=numberOfSteps,
    numberOfProdCycles=numberOfSteps,
    sampleFrequency=100,
    seed=120,
    logLevel=1,
)

# Run the Monte Carlo simulation
mc.run()


chemicalPotential = np.mean(mc.totalChemicalPotentials)

# start refactor
chemicalPotentialError = None
# end refactor

pressures = np.mean(mc.pressures)

In [ ]:
print(f"Chemical potential: {chemicalPotential} +/- {chemicalPotentialError}")

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

## Question 3
It is hard to calculate the chemical potential using Widom particle insertion at higher densities, because the inserted particle has a higher probability of overlapping with another particle. It can be beneficial to sample multiple times for each state of the system. 

How many trial positions are tested in `MonteCarlo::computeChemicalPotential`? 

Another way to improve the calculation is by performing biased insertions using Configurational Bias Monte Carlo (CBMC). In CBMC a particle insertion is done by probing multiple trial positions and orientations. Using expanded ensembles and thermodynamics integration are even better way to calculate the chemical potential.  All these methods will be investigated more in depth in the next week.